# 02 - PDF file normalization

* The notebook's goal is to obtain .txt files from PDFs (via OCR) and normalize them.

* We currently have the official PDF transcripts, which follow strict formatting patterns but have major (yet consistent) changes over time.

    > *The transcripts include a lot of information we don't need: title page, summaries, dates, page numbers...*  
    > *Most of that will be removed during cleaning.*

There are three main format variations, which need different normalization processes:
- **Format 1**: Documents (1, I) to (286, VI)
    - Remove page headers
    - Join lines removing hyphen separators
- **Format 2**: Documents (1, VII) to (53, X)
    - Remove headers and page numbers
    - Join lines removing hyphen separators
- **Format 3**: Documents (55, X) onward
    - Remove headers (different format) and final codes
    - Join lines

> The PDF to text was handled using *pypdf*:   
> ***Versions:***
  "python": "3.12.2",
  "pypdf": "4.1.0"


In [ ]:
from pathlib import Path
import os

# Change input and output directories HERE:
PDF_DIR = Path('..') / 'data' / 'raw' / 'pdfs'
OUTPUT_DIR = Path('..') / 'data' / 'raw' / 'normalized_texts'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [7]:
from pypdf import PdfReader

# Extract text from a PDF file and return a list of strings (one per page).
def process_pdf(path: Path) -> list:
    try:
        reader = PdfReader(str(path))
        pages = reader.pages
        return [page.extract_text() for page in pages]
    except Exception as e:
        print(f'Error reading {path}: {e}')
        return []

In [8]:
# Given a list of lines, clean and concatenate them into a single string, handling hyphenation.
def text_union(lines: list) -> str:
    out = ''
    for line in lines:
        line = line.strip()
        if not line:
            continue
        out += line
        
        # Hyphenation handling: if the line ends with a hyphen, remove it and do not add a space; otherwise, add a space.
        if not out.endswith('-'):
            out += ' '
        else:
            out = out[:-1] 
            
    return out.strip()

In [9]:
def clean_first(pages: list) -> str:
    """
    Normalize format 1 (terms 1-6, num <= 286).
    
    1. Ignore first page (cover)
    2. Delete headers (first short lines)
    3. Join lines handling hyphening
    """
    if not pages:
        return ''
    pages = pages[1:]  # Skip cover page
    frases = []
    for page in pages:
        lines = page.split('\n')
        # Delete headers: líneas cortas al inicio
        while lines and len(lines[0].strip()) < 10:
            lines = lines[1:]
        # Saltar al menos una línea más (separador)
        if lines:
            lines = lines[1:]
        frases.extend(lines)
    return text_union(frases)

In [10]:
def clean_second(pages: list) -> str:
    """
    Normalize format 2 (terms 7 (full term) - 10, num < 54).

    1. Ignore first page (cover)
    2. Delete headers of each page
    3. Delete page numbers at the end
    4. Join lines handling hyphens
    """
    import re
    if not pages:
        return ''
    pages = pages[1:]
    frases = []
    for page in pages:
        lines = page.split('\n')
        if lines:
            lines = lines[1:]  # Delete header (first line)
        if lines:
            # First line may have page numbers at the start, delete them
            first_line = lines[0]
            # Delete digits at the start of the line and any following whitespace
            match = re.match(r'^\d+\s*', first_line)
            if match:
                first_line = first_line[match.end():]
            if first_line.strip():
                frases.append(first_line)
            lines = lines[1:]
        frases.extend(lines)
    return text_union(frases)

In [11]:
def clean_third(pages: list) -> str:
    """
    Normalize format 3 (term 10, num >= 55).
    
    1. Ignore first page (cover)
    2. Delete header (3 first lines of each page)
    3. Delete codes at end of page
    4. Join lines
    
    """
    if not pages:
        return ''
    pages = pages[1:]
    frases = []
    for page in pages:
        lines = page.split('\n')
        # Delete last line (code) and first 3 (header)
        if lines:
            lines = lines[1:]
        if lines and len(lines) > 0:
            lines = lines[2:]
        if lines and lines[-1].strip() == '':
            lines = lines[:-1]
        frases.extend(lines)
    return text_union(frases)

In [14]:
# Batch process all PDFs in a directory, applying the appropriate cleaning function based on the term and number.

def batch_process(pdf_dir: Path, output_dir: Path):
    if not pdf_dir.exists():
        print(f'PDF directory not found: {pdf_dir}')
        return
    
    output_dir.mkdir(parents=True, exist_ok=True)
    failed = []
    success = 0
    
    for pdf_file in sorted(pdf_dir.glob('PL_*.pdf')):
        try:
            # Get term and number from filename
            stem = pdf_file.stem  # e.g., 'PL_1_24'
            parts = stem.split('_')
            if len(parts) < 3:
                print(f'  Skipped (invalid name): {pdf_file.name}')
                continue
            legislatura = int(parts[1])
            numero = int(parts[2])
            
            # Choose cleaning function based on term and number
            pages = process_pdf(pdf_file)
            if not pages:
                print(f'  Failed to extract text: {pdf_file.name}')
                failed.append((legislatura, numero))
                continue
            
            if legislatura <= 6:
                normalized = clean_first(pages)
            elif ( legislatura == 10 and numero < 54 ) or legislatura < 10:
                normalized = clean_second(pages)
            else:
                normalized = clean_third(pages)
            
            output_file = output_dir / f'PL_{legislatura}_{numero}.txt'
            with open(output_file, 'w', encoding='utf-8') as f:
                f.write(normalized)
            success += 1
        except Exception as e:
            print(f'  Error processing {pdf_file.name}: {e}')
            failed.append((legislatura, numero))
    
    print()
    print(f' ✓ Processed {success} files successfully')
    if failed:
        print(f' ✗ Failed {len(failed)} files:')
        for leg, num in failed[:10]:  
            print(f'    PL_{leg}_{num}')
        if len(failed) > 10:
            print(f'    ... and {len(failed) - 10} more')

In [13]:
# Process batch
batch_process(PDF_DIR, OUTPUT_DIR)


 ✓ Processed 19 files successfully
